# Financial Forecast Training Notebook

This notebook trains the updated forecasting pipeline on `income.csv` and `expenses.csv`, then generates forecast outputs, quality scorecards, and visuals.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image

from financial_forecast import build_report, save_dashboard_images, EXPENSE_CATEGORIES


In [ ]:
INCOME_PATH = Path('income.csv')
EXPENSES_PATH = Path('expenses.csv')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Change this to any horizon you want (for example: 3, 6, 12)
MONTHS_AHEAD = 6

INCOME_PATH, EXPENSES_PATH, OUTPUT_DIR, MONTHS_AHEAD


In [ ]:
# Train models + forecast horizon
report_text, chart_data, forecast_daily, training_metrics = build_report(
    INCOME_PATH,
    EXPENSES_PATH,
    months_ahead=MONTHS_AHEAD,
)

# Save dashboard visuals and core files
save_dashboard_images(chart_data, OUTPUT_DIR)
(OUTPUT_DIR / 'forecast_report.txt').write_text(report_text, encoding='utf-8')
(OUTPUT_DIR / 'plain_language_summary.txt').write_text(report_text, encoding='utf-8')
(OUTPUT_DIR / 'dashboard_data.json').write_text(json.dumps(chart_data, indent=2), encoding='utf-8')

# Save daily horizon and next-month subset
forecast_daily['date'] = pd.to_datetime(forecast_daily['date'], errors='coerce')
daily_horizon_name = f'forecast_daily_{MONTHS_AHEAD}m.csv'
forecast_daily.to_csv(OUTPUT_DIR / daily_horizon_name, index=False)

first_forecast_month = pd.to_datetime(chart_data['next_month_forecast']['month'] + '-01')
first_month_mask = forecast_daily['date'].dt.to_period('M') == first_forecast_month.to_period('M')
forecast_daily[first_month_mask].to_csv(OUTPUT_DIR / 'forecast_daily_next_month.csv', index=False)

# Save monthly horizon table (including uncertainty ranges)
monthly_rows = []
for month_key, payload in chart_data['forecast_monthly'].items():
    row = {
        'month': month_key,
        'predicted_income': payload['income'],
        'predicted_income_low': payload.get('income_low', payload['income']),
        'predicted_income_high': payload.get('income_high', payload['income']),
        'predicted_expenses': payload['expenses'],
        'predicted_expenses_low': payload.get('expenses_low', payload['expenses']),
        'predicted_expenses_high': payload.get('expenses_high', payload['expenses']),
        'predicted_profit': payload['profit'],
        'predicted_profit_low': payload.get('profit_low', payload['profit']),
        'predicted_profit_high': payload.get('profit_high', payload['profit']),
        'predicted_margin': payload['margin'],
        'predicted_margin_low': payload.get('margin_low', payload['margin']),
        'predicted_margin_high': payload.get('margin_high', payload['margin']),
    }
    for category in EXPENSE_CATEGORIES:
        row[f'predicted_{category.lower()}'] = payload['expenses_by_category'].get(category, 0.0)
    monthly_rows.append(row)

forecast_monthly_df = pd.DataFrame(monthly_rows)
monthly_horizon_name = f'forecast_monthly_{MONTHS_AHEAD}m.csv'
forecast_monthly_df.to_csv(OUTPUT_DIR / monthly_horizon_name, index=False)

# Save training diagnostics
training_metrics.to_csv(OUTPUT_DIR / 'model_training_metrics.csv', index=False)
model_scorecard_df = pd.DataFrame(chart_data.get('model_scorecard', []))
if not model_scorecard_df.empty:
    model_scorecard_df.to_csv(OUTPUT_DIR / 'model_quality_scorecard.csv', index=False)

h = chart_data.get('forecast_horizon_summary', {})
executive_cards_df = pd.DataFrame([
    {'metric': 'income_total', 'value': h.get('income_total'), 'low': h.get('income_total_low'), 'high': h.get('income_total_high')},
    {'metric': 'expenses_total', 'value': h.get('expenses_total'), 'low': h.get('expenses_total_low'), 'high': h.get('expenses_total_high')},
    {'metric': 'profit_total', 'value': h.get('profit_total'), 'low': h.get('profit_total_low'), 'high': h.get('profit_total_high')},
    {'metric': 'margin_pct', 'value': h.get('margin'), 'low': h.get('margin_low'), 'high': h.get('margin_high')},
    {'metric': 'risk_level', 'value': h.get('risk_level'), 'low': '', 'high': ''},
])
executive_cards_df.to_csv(OUTPUT_DIR / 'forecast_executive_scorecard.csv', index=False)

print('Training + forecast complete.')
print(f'Outputs saved to: {OUTPUT_DIR.resolve()}')
print('- forecast_report.txt')
print('- plain_language_summary.txt')
print('- dashboard_data.json')
print('- forecast_daily_next_month.csv')
print(f'- {daily_horizon_name}')
print(f'- {monthly_horizon_name}')
print('- model_training_metrics.csv')
if not model_scorecard_df.empty:
    print('- model_quality_scorecard.csv')
print('- forecast_executive_scorecard.csv')


In [ ]:
display(Markdown('## Forecast Report'))
print(report_text)


In [ ]:
display(Markdown('## Selected Model Per Target'))
selected_cols = [
    'target',
    'candidate_model',
    'validation_mae',
    'validation_wape',
    'validation_r2',
    'ensemble_member',
    'reliability',
]
selected = training_metrics.loc[training_metrics['selected_model'] == 'YES', selected_cols].reset_index(drop=True)
display(selected)


In [ ]:
display(Markdown('## Model Quality Scorecard (Selected Targets)'))
if 'model_scorecard_df' in locals() and not model_scorecard_df.empty:
    display(model_scorecard_df)
else:
    print('No model scorecard generated.')

display(Markdown('## Full Candidate Training Metrics'))
display(training_metrics)


In [ ]:
display(Markdown('## Horizon KPI Summary (With Ranges)'))
h = chart_data['forecast_horizon_summary']
kpis_df = pd.DataFrame([
    {'metric': 'months_ahead', 'value': h['months_ahead'], 'low': '', 'high': ''},
    {'metric': 'from', 'value': h['from'], 'low': '', 'high': ''},
    {'metric': 'to', 'value': h['to'], 'low': '', 'high': ''},
    {'metric': 'income_total', 'value': h['income_total'], 'low': h.get('income_total_low', ''), 'high': h.get('income_total_high', '')},
    {'metric': 'expenses_total', 'value': h['expenses_total'], 'low': h.get('expenses_total_low', ''), 'high': h.get('expenses_total_high', '')},
    {'metric': 'profit_total', 'value': h['profit_total'], 'low': h.get('profit_total_low', ''), 'high': h.get('profit_total_high', '')},
    {'metric': 'margin_percent', 'value': h['margin'], 'low': h.get('margin_low', ''), 'high': h.get('margin_high', '')},
    {'metric': 'risk_level', 'value': h.get('risk_level', 'N/A'), 'low': '', 'high': ''},
])
display(kpis_df)

display(Markdown('## Executive Scorecard File Preview'))
display(executive_cards_df)


In [ ]:
display(Markdown('## Monthly Forecast Breakdown'))
display(forecast_monthly_df)


In [ ]:
display(Markdown('## Daily Forecast Preview'))
display(forecast_daily.head(10))
display(forecast_daily.tail(10))


In [ ]:
display(Markdown('## Generated Dashboard Images'))
image_names = [
    'income_trend_forecast.png',
    'expenses_by_category_forecast.png',
    'profit_trend_forecast.png',
    'revenue_mix_share.png',
    'kpi_cards_forecast.png',
    'model_quality_scorecard.png',
    'forecast_dashboard.png',
]

for name in image_names:
    path = OUTPUT_DIR / name
    display(Markdown(f'### {name}'))
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f'Missing: {path}')
